# Filter FASTA Files

Subset fasta files such that they only contain shared organisms

In [2]:
import os

In [3]:
def GetCommonSpecies(fasta_files):
    """Gets the species shared across all fasta files
    Make sure your fasta files are saved with genus <space> species in the header

    Args:
        fasta_files (List[str]): List of filepaths to each fasta file

    Returns:
        List[str]: list of shared species names (genus and species)
    """
    # Save species names
    species_names = []

    # For each file, get species
    for file in fasta_files:
        extracted_words = []

        with open(file, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith(">"):
                    
                    # Get parts of the string
                    parts = line.split()
                    
                    # Get the last two parts (genus and species)
                    last_two = parts[-2:]
                    extracted_words.append(" ".join(last_two))
        
        species_names.append(extracted_words)

    shared_species = set.intersection(*map(set, species_names))
    return shared_species


def SubsetFastaFiles(fasta_files, outfile_dir):
    shared_species = GetCommonSpecies(fasta_files)
    print(f"{len(shared_species)} species shared in total.")
    
    for file in fasta_files:
        original_filename = os.path.splitext(os.path.basename(file))[0]
        output = f"{outfile_dir}/subset_{original_filename}_conserved.fasta"
        
        with open(file, 'r') as infile, open(output, 'w') as outfile:
            header = None
            sequence = ""
            
            # read line-by-line
            for line in infile:
                line = line.strip()
                
                # find header!
                if line.startswith(">"):
                    
                    # if we come across a new species, write seq and header to file
                    if header:
                        # Check if the previous sequence's species was conserved
                        if header_species in shared_species:
                            outfile.write(header + '\n')
                            outfile.write(sequence + '\n')
                            
                    # set new header and empty sequence
                    header = line
                    sequence = ""
                    
                    # get genus/species
                    parts = header.split()
                    header_species = " ".join(parts[-2:])
 
                elif header:
                    sequence += line
            
            # Check the last sequence in the file
            if header and header_species in shared_species:
                outfile.write(header + '\n')
                outfile.write(sequence + '\n')

        print(f"Successfully created {output} containing conserved species from {file}")
    

In [13]:
# Fasta filepaths
base_filepath = "./Data/"
all_fastas = [base_filepath+f for f in os.listdir(base_filepath) if f.endswith('.fasta')]
print(all_fastas)

['./Data/gyrA_cds_all_bacteria.fasta', './Data/16s_rRNA_all_bacteria.fasta']


In [16]:
output_dir = "./Data"
SubsetFastaFiles(all_fastas, output_dir)

47 species shared in total.
Successfully created ./Data/subset_gyrA_cds_all_bacteria_conserved.fasta containing conserved species from ./Data/gyrA_cds_all_bacteria.fasta
Successfully created ./Data/subset_16s_rRNA_all_bacteria_conserved.fasta containing conserved species from ./Data/16s_rRNA_all_bacteria.fasta


In [ ]:
# only subset bla gene
all_fastas = ['./Data/related/16s_rRNA_all_bacteria.fasta', './Data/related/bla_cds_all_bacteria.fasta']

In [7]:
output_dir = "./Data/related/bla_subset"
SubsetFastaFiles(all_fastas, output_dir)

24 species shared in total.
Successfully created ./Data/related/bla_subset/subset_16s_rRNA_all_bacteria_conserved.fasta containing conserved species from ./Data/related/16s_rRNA_all_bacteria.fasta
Successfully created ./Data/related/bla_subset/subset_bla_cds_all_bacteria_conserved.fasta containing conserved species from ./Data/related/bla_cds_all_bacteria.fasta


In [4]:
output_dir = "./Data/related/gyrA_subset"
all_fastas = ['./Data/related/16s_rRNA_all_bacteria.fasta', './Data/related/gyrA_cds_all_bacteria.fasta']

SubsetFastaFiles(all_fastas, output_dir)

57 species shared in total.
Successfully created ./Data/related/gyrA_subset/subset_16s_rRNA_all_bacteria_conserved.fasta containing conserved species from ./Data/related/16s_rRNA_all_bacteria.fasta
Successfully created ./Data/related/gyrA_subset/subset_gyrA_cds_all_bacteria_conserved.fasta containing conserved species from ./Data/related/gyrA_cds_all_bacteria.fasta
